In [3]:
import random

# Card class represents a single UNO card with a color and value
class Card:
    # four possible colors in this version of UNO
    COLORS = ["Red", "Blue", "Green", "Yellow"]

    # store the color and value when a card is created
    def __init__(self, color, value):
        self.color = color
        self.value = value

    # check if this card is a Skip by looking at its value
    def isSkip(self):
        return self.value == "Skip"

    # print card as "Color Value" e.g. Red 5 or Blue Skip
    def __repr__(self):
        return f"{self.color} {self.value}"

    # two cards are equal only if both color and value are the same
    def __eq__(self, other):
        return self.color == other.color and self.value == other.value

    # hash allows cards to be stored in sets or used as dictionary keys
    def __hash__(self):
        return hash((self.color, self.value))


# build a full deck of 44 cards (10 numbers + 1 Skip per color) and shuffle it
def generateDeck():
    deck = []
    # loop through each color and add cards 0 to 9 plus one Skip
    for color in Card.COLORS:
        for num in range(10):
            deck.append(Card(color, num))
        # add one Skip card per color
        deck.append(Card(color, "Skip"))
    # shuffle so the order is random at the start of every game
    random.shuffle(deck)
    return deck


# find all cards in a player's hand that can legally be played on the top card
# a card is valid if it matches the top card's color OR its value/number
def getValidMoves(hand, topCard):
    valid = []
    for card in hand:
        # match by color or match by value (includes Skip matching Skip)
        if card.color == topCard.color or card.value == topCard.value:
            valid.append(card)
    return valid


# pack the entire game situation into one dictionary called state
# hands is a list of 3 hands, topCard is the last played card
def makeState(hands, topCard, deck, currentPlayer):
    return {
        "hands": [list(h) for h in hands],
        "topCard": topCard,
        "deck": list(deck),
        "currentPlayer": currentPlayer,
    }


# return a deep copy of the state so simulations don't alter the real game
def cloneState(state):
    return {
        # copy each hand separately so changes in the clone don't affect the original
        "hands": [list(h) for h in state["hands"]],
        "topCard": state["topCard"],
        "deck": list(state["deck"]),
        "currentPlayer": state["currentPlayer"],
    }


# move to the next player in a 3-player rotation (0 -> 1 -> 2 -> 0)
# if skip is True, jump over one player (the one right after current)
def nextPlayer(current, skip=False):
    nxt = (current + 1) % 3
    # skip one more step if a Skip card was played
    if skip:
        nxt = (nxt + 1) % 3
    return nxt


# apply a move to a cloned state and return the resulting new state
# card=None means the player draws from the deck instead of playing
def applyMove(state, playerIdx, card):
    # always work on a copy so the original state is untouched
    s = cloneState(state)
    skipTriggered = False

    if card is None:
        # player has no valid move so they draw one random card from the deck
        if s["deck"]:
            drawn = random.choice(s["deck"])
            s["deck"].remove(drawn)
            s["hands"][playerIdx].append(drawn)
        # after drawing, turn passes normally to the next player
        s["currentPlayer"] = nextPlayer(playerIdx)
    else:
        # remove the chosen card from the player's hand
        s["hands"][playerIdx].remove(card)
        # put it on top of the discard pile
        s["topCard"] = card
        # if a Skip was played, set the flag so the next player is skipped
        if card.isSkip():
            skipTriggered = True
        s["currentPlayer"] = nextPlayer(playerIdx, skip=skipTriggered)

    return s


# the game ends as soon as any player empties their hand
def isTerminal(state):
    for hand in state["hands"]:
        if len(hand) == 0:
            return True
    return False


# return the index of the player who has no cards left, or None if still ongoing
def getWinner(state):
    for i, hand in enumerate(state["hands"]):
        if len(hand) == 0:
            return i
    return None


# heuristic scoring function used at leaf nodes to estimate how good a state is
# higher score means the state is better for playerIdx
def evaluate(state, playerIdx, strategy="defensive"):
    hand = state["hands"][playerIdx]
    # get the indices of both opponents
    oppIdx = [i for i in range(3) if i != playerIdx]
    # average number of cards held by opponents
    oppCards = sum(len(state["hands"][i]) for i in oppIdx) / 2
    # count how many Skip cards the AI is holding (useful for blocking)
    skipCount = sum(1 for c in hand if c.isSkip())
    cAI = len(hand)

    # defensive: punish holding cards heavily, reward having Skip cards to block
    if strategy == "defensive":
        score = 50 - 6 * cAI + 1.5 * oppCards + 4 * skipCount
    # offensive: play aggressively, reward opponents having many cards
    elif strategy == "offensive":
        score = 50 - 4 * cAI + 3 * oppCards + 2 * skipCount
    # balanced fallback used if no strategy is specified
    else:
        score = 50 - 5 * cAI + 2 * oppCards + 3 * skipCount

    # add a large bonus if the AI has already won (empty hand)
    if len(hand) == 0:
        score += 200
    # subtract a large penalty if any opponent has already won
    for i in oppIdx:
        if len(state["hands"][i]) == 0:
            score -= 200

    return score


# minimax with alpha-beta pruning used by Player 1 (defensive) and Player 3
# maximizing=True when it is our player's turn, False when an opponent is moving
# alpha tracks the best score the maximizer can guarantee
# beta tracks the best score the minimizer can guarantee
def minimax(state, depth, playerIdx, maximizing, strategy="defensive", treeLog=None, alpha=float("-inf"), beta=float("inf")):
    if treeLog is None:
        treeLog = []

    # base case: stop recursing when depth runs out or game is over
    if depth == 0 or isTerminal(state):
        val = evaluate(state, playerIdx, strategy)
        treeLog.append({"depth": depth, "type": "LEAF", "value": val})
        return val, None, treeLog

    current = state["currentPlayer"]
    hand = state["hands"][current]
    valid = getValidMoves(hand, state["topCard"])
    # if no valid cards exist the only option is to draw (represented as None)
    moves = valid if valid else [None]

    if maximizing:
        # our player's turn: pick the move with the highest score
        bestVal = float("-inf")
        bestMove = None
        nodeLog = {"depth": depth, "type": "MAX", "player": current, "children": []}
        for move in moves:
            newState = applyMove(state, current, move)
            # check if after this move it is still our player's turn
            childMax = (newState["currentPlayer"] == playerIdx)
            val, _, childLog = minimax(newState, depth - 1, playerIdx, childMax, strategy, [], alpha, beta)
            nodeLog["children"].append({"move": str(move), "value": val, "subtree": childLog})
            if val > bestVal:
                bestVal = val
                bestMove = move
            # update alpha and prune if the minimizer already has a better option
            alpha = max(alpha, bestVal)
            if beta <= alpha:
                break
        treeLog.append(nodeLog)
        return bestVal, bestMove, treeLog
    else:
        # opponent's turn: they will pick the move that hurts our player the most
        bestVal = float("inf")
        bestMove = None
        nodeLog = {"depth": depth, "type": "MIN", "player": current, "children": []}
        for move in moves:
            newState = applyMove(state, current, move)
            childMax = (newState["currentPlayer"] == playerIdx)
            val, _, childLog = minimax(newState, depth - 1, playerIdx, childMax, strategy, [], alpha, beta)
            nodeLog["children"].append({"move": str(move), "value": val, "subtree": childLog})
            if val < bestVal:
                bestVal = val
                bestMove = move
            # update beta and prune if the maximizer already has a better option elsewhere
            beta = min(beta, bestVal)
            if beta <= alpha:
                break
        treeLog.append(nodeLog)
        return bestVal, bestMove, treeLog


# expectimax used by Player 2 (offensive)
# MAX node when it is our player's turn
# OPP (chance) node when an opponent moves, treating all their moves as equally likely
def expectimax(state, depth, playerIdx, treeLog=None):
    if treeLog is None:
        treeLog = []

    # base case: return heuristic score when depth runs out or game is over
    if depth == 0 or isTerminal(state):
        val = evaluate(state, playerIdx, strategy="offensive")
        treeLog.append({"depth": depth, "type": "LEAF", "value": val})
        return val, None, treeLog

    current = state["currentPlayer"]
    hand = state["hands"][current]
    valid = getValidMoves(hand, state["topCard"])

    if current == playerIdx:
        # MAX node: our player picks the action with the highest expected value
        nodeLog = {"depth": depth, "type": "MAX", "player": current, "children": []}
        bestVal = float("-inf")
        bestMove = None

        # evaluate each playable card
        for move in valid:
            newState = applyMove(state, current, move)
            val, _, childLog = expectimax(newState, depth - 1, playerIdx, [])
            nodeLog["children"].append({"move": str(move), "value": val, "subtree": childLog})
            if val > bestVal:
                bestVal = val
                bestMove = move

        # always consider drawing even when valid moves exist
        # drawing is evaluated as a proper chance node over the full deck
        if state["deck"]:
            chanceVal = chanceNode(state, current, depth, playerIdx)
            nodeLog["children"].append({"move": "Draw (Chance)", "value": chanceVal, "type": "CHANCE", "subtree": []})
            if chanceVal > bestVal:
                bestVal = chanceVal
                # None means the best action is to draw
                bestMove = None

        treeLog.append(nodeLog)
        return bestVal, bestMove, treeLog

    else:
        # OPP node: we do not know what the opponent will play
        # assume all their legal moves are equally probable and take the average
        nodeLog = {"depth": depth, "type": "OPP", "player": current, "children": []}
        moves = valid if valid else [None]
        # equal probability for each possible opponent move
        prob = 1.0 / len(moves)
        total = 0.0
        for move in moves:
            newState = applyMove(state, current, move)
            val, _, childLog = expectimax(newState, depth - 1, playerIdx, [])
            nodeLog["children"].append({"move": str(move), "value": val, "subtree": childLog})
            # accumulate weighted average across all opponent moves
            total += prob * val
        treeLog.append(nodeLog)
        return total, None, treeLog


# compute the expected value of drawing a card from the deck
# every card in the deck is equally likely so we weight each outcome by 1/deck_size
def chanceNode(state, playerIdx, depth, rootPlayer):
    deck = state["deck"]
    # if the deck is empty there is nothing to draw so just evaluate current state
    if not deck:
        return evaluate(state, rootPlayer, "offensive")

    total = 0.0
    # each card in the deck has an equal chance of being drawn
    prob = 1.0 / len(deck)

    for card in deck:
        # simulate drawing this specific card and see what the resulting state is worth
        sim = cloneState(state)
        sim["hands"][playerIdx].append(card)
        sim["deck"].remove(card)
        sim["currentPlayer"] = nextPlayer(playerIdx)
        val, _, _ = expectimax(sim, depth - 1, rootPlayer, [])
        total += prob * val

    return total


# print the search tree in a readable indented format for inspection
def printTree(treeLog, indent=0):
    prefix = "  " * indent
    for node in treeLog:
        ntype = node.get("type", "?")
        # leaf nodes just show the heuristic score
        if ntype == "LEAF":
            print(f"{prefix}[LEAF] value={node['value']:.2f}")
        # MAX MIN and OPP nodes show which player is acting and at what depth
        elif ntype in ("MAX", "MIN", "OPP"):
            print(f"{prefix}[{ntype}] player={node.get('player', '?')} depth={node.get('depth', '?')}")
            for child in node.get("children", []):
                # draw (chance) branches are labeled differently from card plays
                if child.get("type") == "CHANCE":
                    print(f"{prefix}  move={child['move']}  expected={child['value']:.2f}  [CHANCE]")
                else:
                    print(f"{prefix}  move={child['move']}  value={child['value']:.2f}")
                    printTree(child.get("subtree", []), indent + 3)
        elif ntype == "CHANCE":
            print(f"{prefix}[CHANCE] expected={node.get('value', 0):.2f}")


# global search depth used by all three players
DEPTH = 3

# human-readable names for each player shown in printed output
PLAYER_NAMES = {
    0: "Player 1 (Minimax/Defensive)",
    1: "Player 2 (Expectimax/Offensive)",
    2: "Player 3"
}


# Player 1 decision: run minimax over all valid moves and return the best one
def p1Move(state, verbose=True):
    valid = getValidMoves(state["hands"][0], state["topCard"])
    # if no valid cards exist the only option is to draw
    moves = valid if valid else [None]

    if verbose:
        print(f"\n{'='*55}")
        print(f"  Player 1 (Minimax / Defensive) - Turn")
        print(f"  Hand  : {state['hands'][0]}")
        print(f"  Top   : {state['topCard']}")
        print(f"  Valid : {valid if valid else ['Draw']}")
        print(f"{'='*55}")

    bestVal = float("-inf")
    bestMove = None
    allScores = []
    fullTree = []

    for move in moves:
        # simulate the move then run minimax from the resulting state
        newState = applyMove(state, 0, move)
        # after this move check if it is still Player 1's turn
        childMax = (newState["currentPlayer"] == 0)
        val, _, subtree = minimax(newState, DEPTH - 1, 0, childMax, "defensive", [])
        allScores.append((move, val))
        fullTree.append({"move": str(move), "value": val, "subtree": subtree})
        if val > bestVal:
            bestVal = val
            bestMove = move

    if verbose:
        print("  All moves considered:")
        for mv, sc in allScores:
            label = str(mv) if mv else "Draw"
            print(f"    {'Play: ' if mv else ''}{label:25s}  score={sc:.2f}")
        chosen = str(bestMove) if bestMove else "Draw"
        print(f"  Chosen: {chosen}  (score={bestVal:.2f})")

    return bestMove, fullTree


# Player 2 decision: run expectimax over valid moves and always compare against drawing
def p2Move(state, verbose=True):
    valid = getValidMoves(state["hands"][1], state["topCard"])

    if verbose:
        print(f"\n{'='*55}")
        print(f"  Player 2 (Expectimax / Offensive) - Turn")
        print(f"  Hand  : {state['hands'][1]}")
        print(f"  Top   : {state['topCard']}")
        print(f"  Valid : {valid if valid else ['Draw']}")
        print(f"{'='*55}")

    moves = valid if valid else [None]
    allScores = []
    fullTree = []
    bestVal = float("-inf")
    bestMove = None

    # score each playable card using expectimax
    for move in moves:
        newState = applyMove(state, 1, move)
        val, _, subtree = expectimax(newState, DEPTH - 1, 1, [])
        allScores.append((move, val))
        fullTree.append({"move": str(move), "value": val, "subtree": subtree})
        if val > bestVal:
            bestVal = val
            bestMove = move

    # always compute the expected value of drawing and compare it against playing
    if state["deck"]:
        chanceVal = chanceNode(state, 1, DEPTH, 1)
        allScores.append((None, chanceVal))
        if chanceVal > bestVal:
            bestVal = chanceVal
            bestMove = None

    if verbose:
        print("  All moves considered:")
        for mv, sc in allScores:
            label = str(mv) if mv else "Draw (Chance Node)"
            print(f"    {'Play: ' if mv else ''}{label:30s}  expected score={sc:.2f}")
        chosen = str(bestMove) if bestMove else "Draw"
        print(f"  Chosen: {chosen}  (score={bestVal:.2f})")

    return bestMove, fullTree


# Player 3 in manual mode: shows the human their hand and reads keyboard input
def p3MoveManual(state):
    hand = state["hands"][2]
    valid = getValidMoves(hand, state["topCard"])

    print(f"\n{'='*55}")
    print(f"  Your Turn (Player 3)")
    print(f"  Top Card : {state['topCard']}")
    print(f"  Your Hand:")
    for i, c in enumerate(hand):
        print(f"    [{i}] {c}")

    # if no valid moves exist the player must draw and their turn ends
    if not valid:
        print("  No valid moves. You must draw a card.")
        input("  Press Enter to draw...")
        return None

    print(f"  Valid moves: {valid}")
    print(f"  Enter index to play, or 'd' to draw:")

    # keep asking until the player gives valid input
    while True:
        choice = input("  >> ").strip().lower()
        if choice == "d":
            return None
        try:
            idx = int(choice)
            card = hand[idx]
            # make sure the chosen card is actually legal to play
            if card in valid:
                return card
            else:
                print("  That card is not valid. Try again.")
        except (ValueError, IndexError):
            print("  Invalid input.")


# Player 3 in simulation mode: uses the same minimax defensive logic as Player 1
def p3MoveSim(state, verbose=True):
    valid = getValidMoves(state["hands"][2], state["topCard"])

    if verbose:
        print(f"\n{'='*55}")
        print(f"  Player 3 (Minimax / Simulation) - Turn")
        print(f"  Hand  : {state['hands'][2]}")
        print(f"  Top   : {state['topCard']}")
        print(f"  Valid : {valid if valid else ['Draw']}")
        print(f"{'='*55}")

    moves = valid if valid else [None]
    bestVal = float("-inf")
    bestMove = None
    allScores = []
    fullTree = []

    # evaluate each move with minimax and pick the best scoring one
    for move in moves:
        newState = applyMove(state, 2, move)
        childMax = (newState["currentPlayer"] == 2)
        val, _, subtree = minimax(newState, DEPTH - 1, 2, childMax, "defensive", [])
        allScores.append((move, val))
        fullTree.append({"move": str(move), "value": val, "subtree": subtree})
        if val > bestVal:
            bestVal = val
            bestMove = move

    if verbose:
        print("  All moves considered:")
        for mv, sc in allScores:
            label = str(mv) if mv else "Draw"
            print(f"    {'Play: ' if mv else ''}{label:25s}  score={sc:.2f}")
        chosen = str(bestMove) if bestMove else "Draw"
        print(f"  Chosen: {chosen}  (score={bestVal:.2f})")

    return bestMove, fullTree


# print a snapshot of the game: turn number, top card, each player's hand size and deck size
def showGameState(state, turnNum):
    print(f"\n  TURN {turnNum}  |  Top Card: {state['topCard']}")
    for i, hand in enumerate(state["hands"]):
        print(f"  {PLAYER_NAMES[i]:35s}: {len(hand)} cards  {hand}")
    print(f"  Deck remaining: {len(state['deck'])}")


# main game loop: deal cards, pick a starting top card, run turns until someone wins
# simulationMode=True means Player 3 is also an AI
# showTrees=True prints the full search tree after each AI decision
def runGame(simulationMode=True, showTrees=False, maxTurns=200):
    print("\n  UNO AI GAME")
    print("  P1=Minimax(Defensive)  P2=Expectimax(Offensive)")
    print(f"  P3={'Simulation' if simulationMode else 'Manual'}\n")

    deck = generateDeck()
    hands = [[], [], []]
    # deal 5 cards to each player one at a time (rotating)
    for _ in range(5):
        for p in range(3):
            hands[p].append(deck.pop(0))

    # the starting top card must not be a Skip or the first turn breaks
    topCard = deck.pop(0)
    while topCard.isSkip():
        # put the Skip back, reshuffle, and try again
        deck.append(topCard)
        random.shuffle(deck)
        topCard = deck.pop(0)

    state = makeState(hands, topCard, deck, currentPlayer=0)
    gameTreeLog = []
    turn = 1

    # main loop: keep playing until someone wins or the turn limit is reached
    while not isTerminal(state) and turn <= maxTurns:
        showGameState(state, turn)
        current = state["currentPlayer"]

        if current == 0:
            # Player 1 uses minimax with defensive strategy
            move, tree = p1Move(state)
            gameTreeLog.append({"turn": turn, "player": 0, "tree": tree})
            if showTrees:
                print("\n  [Search Tree - Player 1]")
                printTree(tree, indent=2)
            state = applyMove(state, 0, move)

        elif current == 1:
            # Player 2 uses expectimax with offensive strategy
            move, tree = p2Move(state)
            gameTreeLog.append({"turn": turn, "player": 1, "tree": tree})
            if showTrees:
                print("\n  [Search Tree - Player 2]")
                printTree(tree, indent=2)
            state = applyMove(state, 1, move)

        elif current == 2:
            if simulationMode:
                # Player 3 uses minimax in simulation mode
                move, tree = p3MoveSim(state)
                gameTreeLog.append({"turn": turn, "player": 2, "tree": tree})
                if showTrees:
                    print("\n  [Search Tree - Player 3]")
                    printTree(tree, indent=2)
            else:
                # Player 3 is human in manual mode
                move = p3MoveManual(state)
            state = applyMove(state, 2, move)

        turn += 1

    # print final results once the game ends
    print(f"\n{'='*55}")
    w = getWinner(state)
    if w is not None:
        print(f"  {PLAYER_NAMES[w]} WINS in {turn - 1} turns!")
    else:
        print(f"  Game ended after {maxTurns} turns (limit reached).")
    print("  Final hands:")
    for i, hand in enumerate(state["hands"]):
        print(f"    {PLAYER_NAMES[i]:35s}: {len(hand)} cards")

    return gameTreeLog, state


# demo mode: set up one game state and print the full search tree for one P1 and P2 decision
def demoSearchTree():
    print("\n  DEMO: Search Tree (P1 Minimax, depth 3)")

    deck = generateDeck()
    hands = [[], [], []]
    for _ in range(5):
        for p in range(3):
            hands[p].append(deck.pop(0))

    topCard = deck.pop(0)
    while topCard.isSkip():
        deck.append(topCard)
        random.shuffle(deck)
        topCard = deck.pop(0)

    state = makeState(hands, topCard, deck, currentPlayer=0)

    print(f"  Top card: {topCard}")
    print(f"  P1 hand : {state['hands'][0]}")
    valid = getValidMoves(state["hands"][0], topCard)
    print(f"  Valid   : {valid if valid else ['Draw']}\n")

    # run minimax from each possible root move and print the resulting subtree
    for move in (valid if valid else [None]):
        newState = applyMove(state, 0, move)
        childMax = (newState["currentPlayer"] == 0)
        val, _, tree = minimax(newState, DEPTH - 1, 0, childMax, "defensive", [])
        label = str(move) if move else "Draw"
        print(f"  Root move: {label}  -> minimax score={val:.2f}")
        printTree(tree, indent=2)
        print()

    print("\n  DEMO: Search Tree (P2 Expectimax, depth 3)")
    # reuse the same hands and top card so both trees are comparable
    state2 = makeState(hands, topCard, deck, currentPlayer=1)
    print(f"  P2 hand : {state2['hands'][1]}")
    valid2 = getValidMoves(state2["hands"][1], topCard)
    print(f"  Valid   : {valid2 if valid2 else ['Draw']}\n")

    # run expectimax from each possible root move and print the resulting subtree
    for move in (valid2 if valid2 else [None]):
        newState2 = applyMove(state2, 1, move)
        val, _, tree = expectimax(newState2, DEPTH - 1, 1, [])
        label = str(move) if move else "Draw"
        print(f"  Root move: {label}  -> expectimax score={val:.2f}")
        printTree(tree, indent=2)
        print()


# entry point: ask the user which mode to run
print("\nSelect mode:")
print("  1 - Simulation (all 3 AI players)")
print("  2 - Manual     (you play as Player 3)")
print("  3 - Demo search trees only")
choice = input("Enter 1, 2, or 3: ").strip()

if choice == "1":
    showTrees = input("Show search trees each turn? (y/n): ").strip().lower() == "y"
    runGame(simulationMode=True, showTrees=showTrees)
elif choice == "2":
    runGame(simulationMode=False, showTrees=False)
elif choice == "3":
    demoSearchTree()
else:
    # default to simulation if input is unrecognized
    print("Invalid choice. Running simulation by default.")
    runGame(simulationMode=True, showTrees=False)


Select mode:
  1 - Simulation (all 3 AI players)
  2 - Manual     (you play as Player 3)
  3 - Demo search trees only


Enter 1, 2, or 3:  1
Show search trees each turn? (y/n):  n



  UNO AI GAME
  P1=Minimax(Defensive)  P2=Expectimax(Offensive)
  P3=Simulation


  TURN 1  |  Top Card: Yellow 3
  Player 1 (Minimax/Defensive)       : 5 cards  [Red 8, Red 6, Blue 9, Blue 2, Red 9]
  Player 2 (Expectimax/Offensive)    : 5 cards  [Red Skip, Green 1, Yellow 7, Yellow 5, Green 2]
  Player 3                           : 5 cards  [Yellow 1, Yellow 8, Yellow 6, Blue 6, Yellow 2]
  Deck remaining: 28

  Player 1 (Minimax / Defensive) - Turn
  Hand  : [Red 8, Red 6, Blue 9, Blue 2, Red 9]
  Top   : Yellow 3
  Valid : ['Draw']
  All moves considered:
    Draw                       score=20.00
  Chosen: Draw  (score=20.00)

  TURN 2  |  Top Card: Yellow 3
  Player 1 (Minimax/Defensive)       : 6 cards  [Red 8, Red 6, Blue 9, Blue 2, Red 9, Green 7]
  Player 2 (Expectimax/Offensive)    : 5 cards  [Red Skip, Green 1, Yellow 7, Yellow 5, Green 2]
  Player 3                           : 5 cards  [Yellow 1, Yellow 8, Yellow 6, Blue 6, Yellow 2]
  Deck remaining: 27

  Player 2 (Expe

### Comparison of Algorithms
**Minimax (Defensive players – Player 1 & 3)**

* Assumes opponents will always make the worst possible move for you
* Plays it safe and tries to avoid risks
* Uses tricks like saving Skip cards to block others
* Problem: In UNO, opponents don’t always have the ability to hurt you, so this is too cautious

**Expectimax (Offensive player – Player 2)**

* Assumes opponents play randomly or unpredictably
* Looks at the *average outcome* instead of the worst case
* Plays more aggressively and uses cards to win faster
* Considers the randomness of drawing cards from the deck

## Key Differences

* **Minimax:** Plays safe, expects the worst
* **Expectimax:** Plays smart, expects what’s most likely

### Conclusion 

Expectimax works better in UNO because the game involves **luck and randomness**.
Minimax is too defensive, while Expectimax makes smarter, more realistic decisions and helps you win faster.
